# The Attention Row That Masks Everything

**Topic:** Masked attention &nbsp;·&nbsp; **Difficulty:** Medium

Masking in attention is done by adding `-inf` to the disallowed scores before the
softmax. It works, right up until a row is masked *entirely*. Then every entry
is `-inf`, the exponentials are all zero, and you divide zero by zero. The loss
becomes `NaN` and the traceback points at your optimizer.

This happens for real: a padded sequence in a batch can have no real keys at all,
so every one of its query rows is fully masked. A causal mask alone never does
this, because the diagonal keeps position i visible to itself. Implement attention with a causal
mask and a padding mask, find the poisoned rows, and fix them.

## The data (created in the next cell)

- `Q`, `K`, `V` each `(2, 6, 8, 16)`: batch 2, 6 heads, 8 positions, head dim 16.
- `lengths` `(2,)`: the true (unpadded) length of each sequence, and one is `0`.
- Causal masking is required as well as padding.

## Your task

With **NumPy only**:

1. Compute scaled scores `Q @ K^T / sqrt(d_head)`.
2. Build the combined mask: position `j` is visible to query `i` only if
   `j <= i` (causal) **and** `j < lengths[b]` (padding).
3. `naive_out`: set masked scores to `-np.inf`, run the usual
   subtract-the-row-max softmax, and multiply by `V`. Count how many output
   entries are non-finite. (Note: a large finite sentinel like `-1e30` does *not*
   reproduce the bug. It silently returns a uniform distribution over the masked
   positions instead, which is its own kind of wrong.)
4. `safe_out`: the same, but for any query row with no visible keys, output
   zeros instead of running the softmax. Count non-finite entries.
5. Set `answers` to a dict with `"n_dead_rows"` (int, (batch, head, query)
   attention rows with nothing visible), `"n_nonfinite_naive"` (int),
   `"n_nonfinite_safe"` (int),
   `"safe_out_sum"` (float), and `"row_sum_max_err"`, the largest deviation from
   1.0 among the attention rows that *do* have visible keys.

## Evaluation

Compared to the reference using a tolerance of `1e-6 * (1 + |reference|)`,
relative for large values and an absolute `1e-6` floor near zero.
`row_sum_max_err` must also be below `1e-9`. The evaluation cell checks the naive
version really does produce non-finite values and the safe one does not.

In [ ]:
import numpy as np

rng = np.random.default_rng(3101)
B, H, T, Dh = 2, 6, 8, 16
Q = rng.normal(0, 1, (B, H, T, Dh))
K = rng.normal(0, 1, (B, H, T, Dh))
V = rng.normal(0, 1, (B, H, T, Dh))
lengths = np.array([5, 0])          # the second sequence is entirely padding
print(f"Q {Q.shape}, lengths {lengths.tolist()}. Sequence 1 has no real tokens")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the data created above to compute the attention outputs, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("n_dead_rows", [], 48.0, -8.03702471109596, 1e-06),
    ("n_nonfinite_naive", [], 768.0, -128.59239537753535, 1e-06),
    ("n_nonfinite_safe", [], 0.0, 0.0, 1e-06),
    ("safe_out_sum", [], -7.464868458461497, 1.2499027555361637, 1e-06),
    ("row_sum_max_err", [], 2.220446049250313e-16, -3.7178707848917005e-17, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "n_nonfinite_naive" in answers:
    _bad = int(answers["n_nonfinite_naive"]) > 0
    _good = int(answers["n_nonfinite_safe"]) == 0
    _rows = answers["row_sum_max_err"] < 1e-9
    print(f"  {'naive produces NaN':<24} {'ok' if _bad else 'wrong'}  "
          f"({int(answers['n_nonfinite_naive'])} entries)")
    print(f"  {'safe version is clean':<24} {'ok' if _good else 'wrong'}")
    print(f"  {'live rows sum to 1':<24} {'ok' if _rows else 'wrong'}")
    _ok = _ok and _bad and _good and _rows

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

A fully-masked row is `softmax([-inf, -inf, ...])`, which is `0/0`. Subtracting
the row max first does not save you: the max is itself `-inf`, so you compute
`-inf - -inf = nan` before the exponential even runs. The only fix is to detect
those rows and short-circuit them. Every production attention kernel does this,
and it is why `attention_mask` handling is the most common source of `NaN` loss
in the first epoch.

```python
import numpy as np

B, H, T, Dh = Q.shape
scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(Dh)

causal = np.tril(np.ones((T, T), dtype=bool))
pad = np.arange(T)[None, :] < lengths[:, None]          # (B, T)
visible = causal[None, :, :] & pad[:, None, :]          # (B, T_query, T_key)
visible = visible[:, None, :, :]                        # broadcast over heads

masked = np.where(visible, scores, -np.inf)

with np.errstate(invalid="ignore"):
    m = masked.max(axis=-1, keepdims=True)
    e = np.exp(masked - m)
    denom = e.sum(axis=-1, keepdims=True)
    attn_naive = e / denom
naive_out = attn_naive @ V

has_any = visible.any(axis=-1)                          # (B, 1, T_query)
safe_attn = np.where(has_any[..., None], attn_naive, 0.0)
safe_attn = np.nan_to_num(safe_attn, nan=0.0, posinf=0.0, neginf=0.0)
safe_out = safe_attn @ V

live = np.broadcast_to(has_any, attn_naive.shape[:-1])
row_sums = np.nan_to_num(attn_naive, nan=0.0).sum(axis=-1)[live]

answers = {
    "n_dead_rows": int((~np.broadcast_to(has_any, attn_naive.shape[:-1])).sum()),
    "n_nonfinite_naive": int((~np.isfinite(naive_out)).sum()),
    "n_nonfinite_safe": int((~np.isfinite(safe_out)).sum()),
    "safe_out_sum": float(safe_out.sum()),
    "row_sum_max_err": float(np.abs(row_sums - 1.0).max()),
}
```